#### Assignment #1 - Part 1 - Average Salary of Teachers in Chicago
-   Data cleansing 
-   Select the right rows for teachers 
-   Delete na values or impute 
-   Winsorze for outliers and compute summary stats

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mstats
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy
import pandas
import random
import sys

import seaborn
from datetime import datetime

In [3]:
ch = pandas.read_excel(r'data\claim_history.xlsx')
ch

,ID,KIDSDRIV,BIRTH,AGE,HOMEKIDS,YOJ,INCOME,PARENT1,HOME_VAL,MSTATUS,...,TIF,CAR_TYPE,RED_CAR,REVOKED,MVR_PTS,CAR_AGE,URBANICITY,CLM_AMT,CLM_COUNT,EXPOSURE
0,63581743,0,1939-03-16,60.0,0,11.0,67000.0,No,NaN,No,...,11,Minivan,yes,No,3,18.0,Highly Urban/ Urban,0,0,0.189
1,132761049,0,1956-01-21,43.0,0,11.0,91000.0,No,257000.0,No,...,1,Minivan,yes,No,0,1.0,Highly Urban/ Urban,0,0,1.000
2,921317019,0,1951-11-18,48.0,0,11.0,53000.0,No,NaN,No,...,1,Van,yes,No,2,10.0,Highly Urban/ Urban,0,0,1.000
3,727598473,0,1964-03-05,35.0,1,10.0,16000.0,No,124000.0,Yes,...,4,SUV,no,No,3,10.0,Highly Urban/ Urban,0,0,0.828
4,450221861,0,1948-06-05,51.0,0,14.0,NaN,No,306000.0,Yes,...,7,Minivan,yes,No,0,6.0,Highly Urban/ Urban,0,0,0.729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10297,67790126,1,1954-08-13,45.0,2,9.0,165000.0,No,386000.0,Yes,...,15,Minivan,no,No,2,17.0,Highly Urban/ Urban,0,0,1.000
10298,61970712,0,1953-06-17,46.0,0,9.0,107000.0,No,333000.0,Yes,...,6,Panel Truck,no,No,0,1.0,Highly Urban/ Urban,0,0,1.000
10299,849208064,0,1951-06-18,48.0,0,15.0,40000.0,No,171000.0,Yes,...,7,SUV,no,No,0,1.0,Highly Urban/ Urban,0,0,1.000
10300,627828331,0,1948-12-12,50.0,0,7.0,43000.0,No,149000.0,Yes,...,6,Minivan,no,No,0,11.0,Highly Urban/ Urban,0,0,1.000


In [4]:
import statsmodels.formula.api as smf
from scipy import stats

# Create Frequency and binary target
data = ch.copy()
data['Frequency'] = data['CLM_COUNT'] / data['EXPOSURE']
data['Event'] = (data['Frequency'] > 1).astype(int)

# Remove rows with missing values in predictors or target
predictors_list = ['REVOKED', 'URBANICITY', 'CAR_AGE', 'MVR_PTS']
data_clean = data[['Event'] + predictors_list].dropna()

print(f"Data shape: {data_clean.shape}")
print(f"Event distribution:\n{data_clean['Event'].value_counts()}")
print(f"\nFirst few rows:\n{data_clean.head()}")


Data shape: (9662, 5)
Event distribution:
Event
0    7264
1    2398
Name: count, dtype: int64

First few rows:
   Event REVOKED           URBANICITY  CAR_AGE  MVR_PTS
0      0      No  Highly Urban/ Urban     18.0        3
1      0      No  Highly Urban/ Urban      1.0        0
2      0      No  Highly Urban/ Urban     10.0        2
3      0      No  Highly Urban/ Urban     10.0        3
4      0      No  Highly Urban/ Urban      6.0        0


In [5]:
# Forward Selection with Entry Threshold = 0.05
entry_threshold = 0.05
in_model = []
candidates = ['REVOKED', 'URBANICITY', 'CAR_AGE', 'MVR_PTS']
entry_order = []

print("=" * 60)
print("FORWARD SELECTION - Entry Threshold = 0.05")
print("=" * 60)

step = 1
while len(candidates) > 0:
    best_pvalue = entry_threshold
    best_predictor = None
    best_model = None
    
    print(f"\nStep {step}:")
    print(f"Current model: {in_model if in_model else 'Intercept only'}")
    print(f"Candidate predictors: {candidates}")
    
    # Test each candidate predictor
    for pred in candidates:
        test_model = in_model + [pred]
        
        # Build formula with categorical variables
        formula_parts = []
        for var in test_model:
            if var in ['REVOKED', 'URBANICITY']:
                formula_parts.append(f"C({var})")
            else:
                formula_parts.append(var)
        
        formula = "Event ~ " + " + ".join(formula_parts)
        
        try:
            fitted = smf.logit(formula, data=data_clean).fit(disp=0)
            
            # Get p-value for the last added predictor
            if var in ['REVOKED', 'URBANICITY']:
                # For categorical, check the first dummy variable p-value
                pvalue = fitted.pvalues[1]  # First coefficient after intercept
            else:
                pvalue = fitted.pvalues[pred]
            
            print(f"  {pred}: p-value = {pvalue:.6f}")
            
            if pvalue < best_pvalue:
                best_pvalue = pvalue
                best_predictor = pred
                best_model = fitted
        except:
            print(f"  {pred}: Failed to fit")
    
    if best_predictor is not None:
        in_model.append(best_predictor)
        candidates.remove(best_predictor)
        entry_order.append(best_predictor)
        print(f"\n  >> {best_predictor} entered the model (p-value = {best_pvalue:.6f})")
        step += 1
    else:
        print(f"\n  >> No predictor meets entry threshold of {entry_threshold}")
        break

print("\n" + "=" * 60)
print("FORWARD SELECTION RESULTS")
print("=" * 60)
print(f"Predictors in model (in order of entry): {entry_order}")
print(f"Last predictor to enter: {entry_order[-1] if entry_order else 'None'}")

FORWARD SELECTION - Entry Threshold = 0.05

Step 1:
Current model: Intercept only
Candidate predictors: ['REVOKED', 'URBANICITY', 'CAR_AGE', 'MVR_PTS']
  REVOKED: p-value = 0.000000
  URBANICITY: p-value = 0.000000
  CAR_AGE: p-value = 0.000000
  MVR_PTS: p-value = 0.000000

  >> MVR_PTS entered the model (p-value = 0.000000)

Step 2:
Current model: ['MVR_PTS']
Candidate predictors: ['REVOKED', 'URBANICITY', 'CAR_AGE']
  REVOKED: p-value = 0.000000
  URBANICITY: p-value = 0.000000
  CAR_AGE: p-value = 0.000000

  >> URBANICITY entered the model (p-value = 0.000000)

Step 3:
Current model: ['MVR_PTS', 'URBANICITY']
Candidate predictors: ['REVOKED', 'CAR_AGE']
  REVOKED: p-value = 0.000000


C:\Users\abhis\AppData\Local\Temp\ipykernel_19980\2765006495.py:41: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalue = fitted.pvalues[1]  # First coefficient after intercept
C:\Users\abhis\AppData\Local\Temp\ipykernel_19980\2765006495.py:41: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalue = fitted.pvalues[1]  # First coefficient after intercept
C:\Users\abhis\AppData\Local\Temp\ipykernel_19980\2765006495.py:41: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value b

  CAR_AGE: p-value = 0.000000

  >> REVOKED entered the model (p-value = 0.000000)

Step 4:
Current model: ['MVR_PTS', 'URBANICITY', 'REVOKED']
Candidate predictors: ['CAR_AGE']
  CAR_AGE: p-value = 0.000000

  >> CAR_AGE entered the model (p-value = 0.000000)

FORWARD SELECTION RESULTS
Predictors in model (in order of entry): ['MVR_PTS', 'URBANICITY', 'REVOKED', 'CAR_AGE']
Last predictor to enter: CAR_AGE


In [7]:
print("\nFinal Model Log-Likelihood:")
print(f"Log-Likelihood: {best_model.llf:.6f}")


Final Model Log-Likelihood:
Log-Likelihood: -4813.622569


In [8]:
# Predict probability for the specified profile
new_profile = pd.DataFrame({
    'URBANICITY': ['Highly Urban/ Urban'],
    'REVOKED': ['No'],
    'CAR_AGE': [8],
    'MVR_PTS': [1]
})

predicted_prob = best_model.predict(new_profile)
print(f"Profile: URBANICITY='Highly Urban/ Urban', REVOKED='No', CAR_AGE=8, MVR_PTS=1")
print(f"Predicted Probability of Frequency > 1: {predicted_prob[0]:.6f}")

Profile: URBANICITY='Highly Urban/ Urban', REVOKED='No', CAR_AGE=8, MVR_PTS=1
Predicted Probability of Frequency > 1: 0.238575
